# TP 4 — Partie 1 : du fichier au flux

**Contexte :** au TP 2, le suivi des colis arrivait une fois par mois, sous forme de fichiers JSON : un message par colis, avec toutes ses étapes, envoyé une fois le colis livré. On le lisait **en batch** : une lecture complète, un résultat, et c'est terminé.

Les transporteurs proposent maintenant d'envoyer chaque **événement** au moment où il se produit : colis pris en charge, en transit, en livraison, livré, ou incident. Les données n'ont plus de fin. C'est un **flux**.

| | Batch (TP 1 à 3) | Streaming (TP 4) |
|---|---|---|
| Données | un ensemble fini, connu au départ | une suite sans fin d'événements |
| Traitement | lancé à heure fixe, sur tout ou sur un lot | permanent, par petits **micro-lots** |
| Résultat | exact pour les données du lot | mis à jour à chaque micro-lot |

**Objectif de cette partie :** voir qu'une requête Spark Structured Streaming s'écrit presque comme une requête batch. Spark la relance seul à chaque arrivée de données.

> **⚠️ Un seul notebook actif à la fois.** Arrêtez le noyau des autres notebooks avant de commencer. Le profil `streaming` n'est pas nécessaire pour cette partie : elle n'utilise pas encore Kafka.

In [ ]:
import time

from pyspark.sql import SparkSession, functions as F

from flux import CHECKPOINTS, SCHEMA_EVENEMENT, a_completer, arreter_flux, fin_du_rejeu, lister_s3, rejouer, vider_s3

spark = (SparkSession.builder.appName("tp4-fichier-flux")
         .config("spark.ui.showConsoleProgress", "false")   # pas de barres de progression
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")     # n'affiche que les erreurs de Spark
DOSSIER = "s3://landing/flux/suivi"

arreter_flux(spark)          # si la cellule est relancée, on arrête les requêtes précédentes
vider_s3(DOSSIER)            # on repart d'un dossier vide
vider_s3(f"{CHECKPOINTS}/par_code")

## Étape 1 — Faire arriver les événements

Le script `rejouer_suivi.py` relit les messages du TP 2, les découpe en événements et les envoie dans l'ordre où ils se sont produits, en **temps accéléré**. Ici, il écrit **un fichier JSON par jour** dans `landing/flux/suivi/`, au rythme d'un jour toutes les deux secondes (`--jours-par-seconde 0.5`).

Il garde les défauts réels d'un flux : des messages tronqués, des événements en double, et 5 % d'événements qui arrivent avec plusieurs heures de retard.

La cellule lance le rejeu **en arrière-plan** : le notebook reste disponible pendant qu'il tourne.

In [ ]:
rejeu = rejouer("--dossier", DOSSIER, "--depuis", "2018-03-01", "--jusqu-a", "2018-03-11",
                "--jours-par-seconde", 0.5, attendre=False)
while len(lister_s3(DOSSIER)) < 3:      # on attend les trois premiers jours
    time.sleep(1)
for cle, taille in lister_s3(DOSSIER):
    print(f"{cle:45} {taille:>8} octets")

Un événement par ligne. Regardons les deux premières lignes du premier fichier.

In [ ]:
for ligne in spark.read.text("s3a://landing/flux/suivi/suivi-2018-03-01.json").limit(2).collect():
    print(ligne.value)

## Étape 2 — Lire le dossier en continu

En batch, on écrirait `spark.read.json(chemin)`. En streaming, on écrit **`spark.readStream`** : Spark surveille le dossier et lit chaque nouveau fichier dès qu'il apparaît.

Créez le DataFrame `flux` qui lit `s3a://landing/flux/suivi/` en continu, avec :
- le schéma `SCHEMA_EVENEMENT` (`.schema(...)`) ;
- l'option `maxFilesPerTrigger` à `1` : un seul nouveau fichier par micro-lot, pour bien voir les étapes.

In [ ]:
flux = a_completer("flux")  # À COMPLÉTER
print("flux en continu :", flux.isStreaming)
flux.printSchema()

> **❓ Question 1.** Retirez `.schema(...)` et relancez la cellule : que se passe-t-il ? Pourquoi Spark ne peut-il pas deviner le schéma d'un flux comme il le fait en batch ? Remettez ensuite le schéma.

## Étape 3 — Une requête qui se met à jour seule

Sur `flux`, écrivez la même requête qu'en batch : le **nombre d'événements par `code`**, dans un DataFrame `par_code`. Puis la cellule suivante la démarre avec `writeStream` :
- le résultat va dans une table en mémoire nommée `par_code` (`format("memory")`), pratique pour l'observer ;
- le mode `complete` réécrit tout le résultat à chaque micro-lot ;
- un micro-lot est lancé toutes les 3 secondes.

In [ ]:
par_code = a_completer("par_code")  # À COMPLÉTER : nombre d'événements par code
requete = (par_code.writeStream
           .format("memory").queryName("par_code")
           .outputMode("complete")
           .option("checkpointLocation", f"{CHECKPOINTS}/par_code")
           .trigger(processingTime="3 seconds")
           .start())

Observez le résultat grandir pendant que les fichiers arrivent (la cellule dure environ 20 secondes).

In [ ]:
for i in range(4):
    time.sleep(5)
    lots = [p for p in requete.recentProgress if p["numInputRows"] > 0]
    print(f"--- après {5 * (i + 1)} s : {len(lots)} micro-lot(s) avec des données")
    spark.sql("SELECT code, `count` AS evenements FROM par_code ORDER BY evenements DESC").show()

On attend la fin du rejeu, puis que la requête ait lu tous les fichiers, et on l'arrête.

In [ ]:
n_envoyes = fin_du_rejeu(rejeu)
requete.processAllAvailable()
requete.stop()
spark.sql("SELECT code, `count` AS evenements FROM par_code ORDER BY evenements DESC").show()
print("total lu par la requête :", spark.sql("SELECT sum(`count`) FROM par_code").first()[0])

> **❓ Question 2.** Combien de micro-lots ont contenu des données, et pourquoi ce nombre ? Le total lu correspond-il au nombre d'événements envoyés ? Un message tronqué (il y en a quelques-uns par mois) apparaîtrait ici avec un `code` égal à `null` : pourquoi Spark ne s'arrête-t-il pas sur ce JSON invalide ?

In [ ]:
for p in requete.recentProgress:
    if p["numInputRows"] > 0:
        print(f"micro-lot {p['batchId']:>2} : {p['numInputRows']:>5} lignes, "
              f"position dans le dossier : {p['sources'][0]['endOffset']}")

## Étape 4 — Ce qui ne marche pas comme en batch

En streaming, le **mode de sortie** dit ce qui est écrit à chaque micro-lot :

| Mode | Ce qui est écrit | Possible avec une agrégation ? |
|---|---|---|
| `append` | seulement les **nouvelles** lignes, qui ne changeront plus | seulement avec un *watermark* (partie 4) |
| `update` | les lignes **modifiées** par le micro-lot | oui |
| `complete` | **tout** le résultat | oui |

Essayons le mode `append` sur `par_code`.

In [ ]:
try:
    par_code.writeStream.format("memory").queryName("essai").outputMode("append").start()
except Exception as e:
    print(type(e).__name__, ":", str(e).splitlines()[0][:250])

> **❓ Question 3.** Pourquoi Spark refuse-t-il le mode `append` ici ? Pour compter les événements par code depuis le début, que doit-il garder en mémoire d'un micro-lot à l'autre ? Que deviendrait cette mémoire si le flux tournait pendant un an ?

## À retenir

- Une requête en streaming s'écrit comme une requête batch : seuls `readStream` et `writeStream` changent.
- Spark découpe le flux en **micro-lots** et met le résultat à jour à chaque lot.
- Le schéma est obligatoire, et une agrégation oblige Spark à garder un **état** entre les micro-lots.

Dans la partie 2, les fichiers sont remplacés par **Kafka**, l'outil qui transporte les flux en entreprise.